# Peer-to-Peer Dataset Federation Demo
This notebook shows how a researcher can query, from a plain Jupyter notebook, Parquet files spread across several institutions as if they were all local, with no centralized infrastructure.

Two Parquet files are used for this demo, already present in `node/data/`:
- `yellow_tripdata_2026-01.parquet`
- `yellow_tripdata_2026-02.parquet`

They come from the public NYC taxi trip data: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

For the full architecture (Rust node, iroh, gossip, HTTP API), see [system_architecture.md](../../doc/system_architecture.md).

## Prerequisites

Before running this notebook, at leat one Rust node must be running locally (it exposes the HTTP API on `localhost:8080`):

```bash
cd peer-dataset-federation/node
INSTITUTION=<institution> cargo run -- peer
```

See [installation.md](../../doc/installation.md) for instructions on how to launch the project.

## 1. Connecting to the local node

`P2PClient` is the HTTP wrapper to the Rust node. `P2PDataset` is the layer used in this notebook: local cache, file discover, loading into DataFrames.

In [7]:
import logging
import pandas as pd
from p2p.client import P2PClient  # HTTP wrapper: talks to the Rust node
from p2p.dataset import P2PDataset  # Cache + query layer: files(), load(), query(), federate()

# Show INFO-level logs (cache hits/misses, fetch calls) directly in the notebook
logging.basicConfig(level=logging.INFO)

# Connect to the local node
client = P2PClient("http://localhost:8080")

# Main entry point: everything below goes through this object
dataset = P2PDataset(client)

## 2. Exploring the files available on the network

`files()` queries the local node, which answers from the manifests received via gossip (its own + every other peer's). `files_df()` shows the same information as a readable table.

In [8]:
dataset.files()

[{'file_name': 'yellow_tripdata_2026-02.parquet',
  'institution': 'mtl',
  'num_rows': 3399866,
  'num_row_groups': 4,
  'file_size_bytes': 58683353,
  'columns': [{'c_type': 'INT32', 'name': 'VendorID'},
   {'c_type': 'INT64', 'name': 'tpep_pickup_datetime'},
   {'c_type': 'INT64', 'name': 'tpep_dropoff_datetime'},
   {'c_type': 'INT64', 'name': 'passenger_count'},
   {'c_type': 'DOUBLE', 'name': 'trip_distance'},
   {'c_type': 'INT64', 'name': 'RatecodeID'},
   {'c_type': 'BYTE_ARRAY', 'name': 'store_and_fwd_flag'},
   {'c_type': 'INT32', 'name': 'PULocationID'},
   {'c_type': 'INT32', 'name': 'DOLocationID'},
   {'c_type': 'INT64', 'name': 'payment_type'},
   {'c_type': 'DOUBLE', 'name': 'fare_amount'},
   {'c_type': 'DOUBLE', 'name': 'extra'},
   {'c_type': 'DOUBLE', 'name': 'mta_tax'},
   {'c_type': 'DOUBLE', 'name': 'tip_amount'},
   {'c_type': 'DOUBLE', 'name': 'tolls_amount'},
   {'c_type': 'DOUBLE', 'name': 'improvement_surcharge'},
   {'c_type': 'DOUBLE', 'name': 'total_amou

In [9]:
dataset.files_df()

,file_name,institution,num_rows,num_row_groups,size,columns
0,yellow_tripdata_2026-02.parquet,mtl,3399866,4,56.0 MB,"VendorID, tpep_pickup_datetime, tpep_dropoff_d..."
1,yellow_tripdata_2026-01.parquet,mtl,3724889,4,61.2 MB,"VendorID, tpep_pickup_datetime, tpep_dropoff_d..."


In [10]:
# max_columns_shown limits how many columns are shown per file
dataset.files_df(2)

,file_name,institution,num_rows,num_row_groups,size,columns
0,yellow_tripdata_2026-02.parquet,mtl,3399866,4,56.0 MB,"VendorID, tpep_pickup_datetime, ... (+18 more)"
1,yellow_tripdata_2026-01.parquet,mtl,3724889,4,61.2 MB,"VendorID, tpep_pickup_datetime, ... (+18 more)"


## 3. Loading a single file: `load()`

`load()` fetches the file (from the local cache if already downloaded, otherwise over the network) and loads it directly into a `pandas.DataFrame`.

In [11]:
df = dataset.load("yellow_tripdata_2026-01.parquet")
df.head()

INFO:p2p.dataset:cache hit: yellow_tripdata_2026-01.parquet
INFO:p2p.dataset:loading 'yellow_tripdata_2026-01.parquet' into dataframe


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


## 4. Loading several files independently: `query()`

`query()` fetches several files at once and returns one DataFrame per file (no merging between them).

In [12]:
results = dataset.query("yellow_tripdata_2026-01.parquet", "yellow_tripdata_2026-02.parquet")

for name, df in results.items():
    print(name, df.shape)

INFO:p2p.dataset:cache hit: yellow_tripdata_2026-01.parquet
INFO:p2p.dataset:cache hit: yellow_tripdata_2026-02.parquet


yellow_tripdata_2026-01.parquet (3724889, 20)
yellow_tripdata_2026-02.parquet (3399866, 20)


## 5. Querying several files together with SQL: `federate()`

`federate()` fetches several files and exposes them together as a single DuckDB view name `dataset`. You can then write a single SQL query over all of them, without worrying about where each file comes from.

In [13]:
con = dataset.federate("yellow_tripdata_2026-01.parquet", "yellow_tripdata_2026-02.parquet")

df2 = con.sql('''SELECT * FROM dataset WHERE "passenger_count" > 2''').df()
df2

INFO:p2p.dataset:cache hit: yellow_tripdata_2026-01.parquet
INFO:p2p.dataset:cache hit: yellow_tripdata_2026-02.parquet
INFO:p2p.dataset:federated view 'dataset' created from 2 file(s)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,38.7,1.0,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
1,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3,1.83,1,N,237,263,1,10.7,1.0,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.00
2,2,2026-01-01 00:42:31,2026-01-01 00:44:28,3,0.33,1,N,234,90,2,4.4,1.0,0.5,0.00,0.0,1.0,10.15,2.5,0.0,0.75
3,2,2026-01-01 00:10:51,2026-01-01 00:18:46,4,1.05,1,N,164,164,1,9.3,1.0,0.5,3.01,0.0,1.0,18.06,2.5,0.0,0.75
4,2,2026-01-01 00:28:29,2026-01-01 00:46:04,3,4.78,1,N,238,243,1,23.3,1.0,0.5,2.00,0.0,1.0,27.80,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242507,2,2026-02-28 23:06:44,2026-02-28 23:31:36,3,3.70,1,N,79,140,1,24.0,1.0,0.5,5.95,0.0,1.0,35.70,2.5,0.0,0.75
242508,2,2026-02-28 23:15:21,2026-02-28 23:32:53,3,1.55,1,N,48,170,1,15.6,1.0,0.5,3.20,0.0,1.0,24.55,2.5,0.0,0.75
242509,2,2026-02-28 23:52:49,2026-03-01 00:03:09,3,1.63,1,N,186,107,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75
242510,2,2026-02-28 23:34:43,2026-02-28 23:50:47,3,1.02,1,N,79,249,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75


## 6. A custom function on top of `federate()`: `filter_passenger()`

The goal is to show that project-specific Python functions can be built on top of the federated view, here `filter_passenger()`, so the researcher doesn't need to know SQL for common filters. It's the exact same query as the on written by hand above, hidden behind a plain method call.

In [14]:
df3 = dataset.filter_passenger(con, 3)
df3

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
1,2,2026-01-01 00:10:51,2026-01-01 00:18:46,4,1.05,1,N,164,164,1,9.3,1.00,0.5,3.01,0.0,1.0,18.06,2.5,0.0,0.75
2,1,2026-01-01 00:53:37,2026-01-01 01:11:29,4,3.60,1,N,232,164,2,19.8,4.25,0.5,0.00,0.0,1.0,25.55,2.5,0.0,0.75
3,2,2026-01-01 00:04:06,2026-01-01 00:22:51,4,5.07,1,N,158,151,1,25.4,1.00,0.5,6.23,0.0,1.0,37.38,2.5,0.0,0.75
4,2,2026-01-01 00:33:13,2026-01-01 01:04:31,4,4.05,1,N,142,234,1,29.6,1.00,0.5,0.00,0.0,1.0,35.35,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112122,2,2026-02-28 23:27:52,2026-02-28 23:33:25,4,0.34,1,N,114,114,1,6.5,1.00,0.5,2.45,0.0,1.0,14.70,2.5,0.0,0.75
112123,2,2026-02-28 23:29:50,2026-02-28 23:46:38,4,2.34,1,N,90,144,1,15.6,1.00,0.5,4.27,0.0,1.0,25.62,2.5,0.0,0.75
112124,2,2026-02-28 23:31:39,2026-02-28 23:50:34,4,2.36,1,N,79,186,1,17.7,1.00,0.5,2.50,0.0,1.0,25.95,2.5,0.0,0.75
112125,2,2026-02-28 23:39:59,2026-02-28 23:54:31,4,1.55,1,N,48,164,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
